# Investigação de similaridade semântica — execução do plano

Resultados dos experimentos de 16/09/2026. Este notebook é independente de
`TCC.ipynb` e não muda suas células. Veja [protocolo](investigacao/README.md),
[rastreabilidade](investigacao/METODOLOGIA.md) e
[relatório](analises/implementacao_similaridade_semantica.md).

**Limite:** questionário e B3 são diagnósticos já conhecidos; documentos não têm
gabarito de posição. O experimento supervisionado externo requer textos da base.
Não há nova anotação manual nem resultados de treinamento externo inventados.

## Configuração e reprodução — R4

Kapoor e Narayanan (2023), *Leakage and the reproducibility crisis in machine-learning-based science*,
DOI 10.1016/j.patter.2023.100804: dados/configurações explícitos. Aqui usamos
manifestos e hashes, adaptação de engenharia descrita em `METODOLOGIA.md`.


In [1]:
import json
from pathlib import Path
from collections import Counter
ROOT = Path.cwd()
if not (ROOT / 'TCC.ipynb').exists():
    raise RuntimeError('Execute o notebook na raiz do repositório TCC.')
RUNS = ROOT / 'analises' / 'execucoes'
def load(run, filename):
    # R4: leitura de resultado persistido, sem recalcular métricas.
    return json.loads((RUNS / run / filename).read_text(encoding='utf-8'))
diag = 'diagnostico_e5_20260916'
doc = 'documento_6x1_20260916'
controlled = 'controlados_20260916'
print(json.dumps(load(diag, 'manifest.json')['config'], ensure_ascii=False, indent=2))


{
  "experiment": "diagnostico",
  "encoder": {
    "name": "intfloat/multilingual-e5-base",
    "revision": "d128750597153bb5987e10b1c3493a34e5a4502a",
    "requested_revision": null,
    "max_length": 512,
    "device": "cpu",
    "encoded": 222,
    "tokens": 5135,
    "truncated": 0
  },
  "seed": 42,
  "prefix": "query",
  "sources": [
    "instrumento",
    "construcao_controlada"
  ]
}


## 1. Direção no instrumento — R1/R2

Chen et al. (2023), *Ideology Prediction from Scarce and Biased Supervision*,
`artigos/2023.acl-long.530.pdf`, §§3–4, e Duan et al. (2025), *Constructing
Vec-tionaries*, arquivo local identificado em `METODOLOGIA.md`, §3.1.
Aplicação: projeções B0/B2 existentes, sem reproduzir treinamento dos artigos.
A margem é proporcional a B0. `effect` é convenção do instrumento, não intensidade.
O prefixo `query:` é aplicado uma vez. A comparação com saídas históricas não é
ablação isolada: também se normalizam espaços e se usa o ambiente/revisão registrados.


In [2]:
metrics = load(diag, 'diagnostico_metricas.json')
print('eixo | método | calibração | n | balanced accuracy | MCC')
for r in metrics:
    if r['method'] != 'margem_equivalente_B0':
        print(f"{r['axis']} | {r['method']} | {r['calibration']} | {r['n']} | {r['balanced_accuracy']:.4f} | {r['mcc']:.4f}")


eixo | método | calibração | n | balanced accuracy | MCC
economic | B0 | zero | 22 | 0.5556 | 0.2623
economic | B0 | B3b_transfer_B5 | 22 | 0.4573 | -0.0944
economic | B2 | zero | 22 | 0.5171 | 0.0585
economic | B2 | B3b_transfer_B5 | 22 | 0.5171 | 0.0585
diplomatic | B0 | zero | 24 | 0.6250 | 0.2582
diplomatic | B0 | B3b_transfer_B5 | 24 | 0.6250 | 0.2582
diplomatic | B2 | zero | 24 | 0.5000 | 0.0000
diplomatic | B2 | B3b_transfer_B5 | 24 | 0.5000 | 0.0000
state | B0 | zero | 37 | 0.6522 | 0.3045
state | B0 | B3b_transfer_B5 | 37 | 0.5224 | 0.0432
state | B2 | zero | 37 | 0.4792 | -0.1227
state | B2 | B3b_transfer_B5 | 37 | 0.5240 | 0.0739
society | B0 | zero | 33 | 0.6611 | 0.3246
society | B0 | B3b_transfer_B5 | 33 | 0.6611 | 0.3246
society | B2 | zero | 33 | 0.5833 | 0.1936
society | B2 | B3b_transfer_B5 | 33 | 0.6222 | 0.2582


## 2. Ordenação e calibração agrupada — R1/R4

Chen et al. (2023), §§3–4, e prevenção de vazamento de R4: lados do mesmo par
ficam juntos no leave-one-pair-out. B5 transfere limiar B3b; não valida limiar
documental. Os pares já foram usados no desenvolvimento.


In [3]:
pairs = load(diag, 'pares_b3_b4.json')
print('eixo | método | ordenação | BA sem calibração | BA leave-one-pair-out')
for r in pairs:
    if r['dataset'] == 'INDEPENDENT_MINIMAL_PAIRS' and r['method'] != 'margem_equivalente_B0':
        print(f"{r['axis']} | {r['method']} | {r['ordering_accuracy']:.3f} | {r['raw']['balanced_accuracy']:.3f} | {r['leave_one_pair_out']['balanced_accuracy']:.3f}")


eixo | método | ordenação | BA sem calibração | BA leave-one-pair-out
economic | B0 | 1.000 | 0.562 | 0.562
economic | B2 | 0.875 | 0.625 | 0.688
diplomatic | B0 | 1.000 | 1.000 | 1.000
diplomatic | B2 | 1.000 | 1.000 | 1.000
state | B0 | 0.750 | 0.500 | 0.688
state | B2 | 0.500 | 0.500 | 0.438
society | B0 | 1.000 | 0.938 | 0.875
society | B2 | 1.000 | 0.812 | 0.812


## 3. Contextos controlados e NLI — R5/R7

Ribeiro et al. (2020), *Beyond Accuracy: Behavioral Testing of NLP Models with
CheckList*, §2 ([fonte](https://aclanthology.org/2020.acl-main.442/)); Yin et al.
(2019), *Benchmarking Zero-shot Text Classification*, §3
([fonte](https://aclanthology.org/D19-1404/)). Templates declaram apoio/oposição;
não se inverte uma frase natural apenas inserindo negação. Expectativa por
construção não é anotação humana. Recall abaixo é só da passagem inserida.


In [4]:
cases = load(controlled, 'templates_nli.json')
contexts = load(controlled, 'contextos.json')
print('Expectativas NLI satisfeitas:', sum(r['satisfied'] for r in cases), '/', len(cases))
print('Erros:', [(r['id'], r['expected'], r['predicted']) for r in cases if not r['satisfied']])
for k in ['1', '3', '5']:
    print('Recall da inserção @' + k, sum(r['retrieval']['recall_inserted'][k] for r in contexts) / len(contexts))


Expectativas NLI satisfeitas: 8 / 8
Erros: []
Recall da inserção @1 1.0
Recall da inserção @3 1.0
Recall da inserção @5 1.0


## 4. Texto 6x1 sem gabarito — R5/R7 e plano §7

As fontes R5/R7 motivam os testes, mas não validam estes limiares. Deduplicação,
cobertura e intervalo algébrico são adaptações específicas do plano. Direção
aceita pelo sistema não determina intensidade. Por isso `response=null` e o
intervalo integral do quiz continua [0,100]; isso não significa centrismo.


In [5]:
result = load(doc, 'documento.json')
print('Estados:', dict(Counter(r['state'] for r in result['base']['items'])))
print('Cobertura direcional declarada:', result['base']['direction_coverage'])
print('Invariâncias e estabilidade:', {k: sum(r[k] for r in result['checks']) for k in result['checks'][0] if k != 'question_id'})
print('Custo:', json.dumps(load(doc, 'manifest.json'), ensure_ascii=False, indent=2))


Estados: {'incerta': 49, 'contraria': 14, 'favoravel': 6, 'conflito': 1}
Cobertura direcional declarada: {'econ': 0.15384615384615385, 'dipl': 0.3111111111111111, 'govt': 0.359375, 'scty': 0.17123287671232876}
Invariâncias e estabilidade: {'spacing_invariant': 70, 'duplicate_invariant': 70, 'aggregation_order_invariant': 70, 'segmentation_same_state': 54}
Custo: {
  "config": {
    "command": "documents",
    "encoder": {
      "name": "intfloat/multilingual-e5-base",
      "revision": "d128750597153bb5987e10b1c3493a34e5a4502a",
      "requested_revision": null,
      "max_length": 512,
      "device": "cpu",
      "encoded": 109,
      "tokens": 9355,
      "truncated": 0
    },
    "nli": {
      "name": "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
      "revision": "8adb042d524ecd5c26d3e3ba0e3fbcf7e2d0864c",
      "requested_revision": null,
      "device": "cpu",
      "pairs": 420,
      "tokens": 114868,
      "truncated": 0,
      "id2label": {
        "0": "entailment",
        "

## 5. Próxima execução externa — R3/R6

Silva e Paraboni (2023), *Politically-oriented information inference from text*,
arquivo local, §§3.2/4.1.2: distingue posição textual e ideologia autoral.
Tunstall et al. (2022), *Efficient Few-Shot Learning Without Prompts*, §3.1
([fonte](https://arxiv.org/abs/2209.11055)): duas etapas, adaptação com pares do
mesmo alvo e cabeça logística L2. Treinamento ainda não executado em corpus externo.

O UstanceBR r3 foi acessado: CC BY 4.0, CSVs com `Tweet_ID;Polarity`, sem textos
ou autoria nesses arquivos. Veja `analises/acesso_corpora.json`. Após obter
textos e autores, o adaptador constrói dev sem usar teste. Não é necessária
anotação manual nova. Execute na raiz:

```powershell
python -m investigacao prepare-ustancebr --archive analises/ustancebr_r3.zip --hydrated dados/tweets_hidratados.jsonl --output dados/ustancebr
python -m investigacao external --corpus dados/ustancebr/corpus.jsonl --metadata dados/ustancebr/corpus.meta.json --output analises/execucoes/externo_novo
python -m investigacao documents --text 'content/Escala 6x1.txt' --selected-run analises/execucoes/externo_novo --output analises/execucoes/transferencia_nova
```

Comparação com Sabiá: acrescente `--sabia` com previsões por pergunta e
configuração fixa, conforme README. Nenhum resultado disponível foi tratado
como Sabiá por inferência. A ausência desse comparador está registrada.
